# Understanding Code Ownership in siege_utilities

**Scenario:** A new team member joins and needs to understand the repo:
which modules change most, what the commit patterns look like, and who
owns what. Rather than reading every file, they use the git/ module to
query the repository's own history programmatically.

## 1. Repository Status: Where Are We?

Before analyzing history, get the current state: which branch, any
uncommitted work, last commit.

In [1]:
import os
from siege_utilities.git.git_status import get_repository_status, get_branch_info

# Point to the repo root (notebook may run from a subdirectory)
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
# Find the actual repo root by walking up until we find .git
p = os.path.abspath('.')
while p != '/' and not os.path.isdir(os.path.join(p, '.git')):
    p = os.path.dirname(p)
repo_root = p

status = get_repository_status(repo_path=repo_root)

print(f"Repository: {status['repository_path']}")
print(f"Branch:     {status['current_branch']}")
print(f"Clean:      {status['working_directory_clean']}")
print(f"Changes:    {status['total_changes']} total")
if status.get('last_commit'):
    lc = status['last_commit']
    print(f"Last commit: {lc.get('hash', 'N/A')[:8]} — {lc.get('message', 'N/A')[:60]}")

Repository: /Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities
Branch:     feat/580-notebook-audit
Clean:      False
Changes:    35 total
Last commit: bf3c475 — Merge pull request #1002 from siege-analytics/feat/816-noteb


## 2. Recent Commit History and Categorization

The branch analyzer parses conventional commits (feat, fix, docs, etc.)
and categorizes them. This tells you what kind of work is active.

In [2]:
from siege_utilities.git.branch_analyzer import get_commit_history, categorize_commits

commits = get_commit_history(limit=20, repo_path=repo_root)
categories = categorize_commits(commits)

print(f"Last {len(commits)} commits by category:")
print(f"{'Category':<15} {'Count':>5}")
print("-" * 22)
for cat, items in sorted(categories.items(), key=lambda x: -len(x[1])):
    if items:
        print(f"{cat:<15} {len(items):>5}")

print(f"\nRecent commits:")
for c in commits[:5]:
    print(f"  {c['hash'][:8]} {c['date'][:10]} {c['message'][:65]}")

Last 20 commits by category:
Category        Count
----------------------
features           16
fixes               4

Recent commits:
  bf3c475 2026-06-03 Merge pull request #1002 from siege-analytics/feat/816-notebook-c
  5b9dae4 2026-06-03 feat(#816): add notebook coverage for economic, files, identifier
  5b71aca 2026-06-03 Merge pull request #1001 from siege-analytics/feat/784-docs-quali
  0ec5627 2026-06-03 feat(#784): add interrogate docstring enforcement and expand sphi
  7e17b23 2026-06-03 Merge pull request #1000 from siege-analytics/feat/780-integratio


## 3. Branch Status: How Far Ahead/Behind?

`analyze_branch_status` shows the relationship between the current
branch and main — useful for knowing if you need to rebase.

In [3]:
from siege_utilities.git.branch_analyzer import analyze_branch_status

branch_status = analyze_branch_status(repo_path=repo_root)

print(f"Branch analysis:")
for key, value in branch_status.items():
    print(f"  {key}: {value}")

Branch analysis:
  branch: feat/580-notebook-audit
  ahead: 0
  behind: 14
  last_commit_hash: bf3c475
  last_commit_date: 2026-06-03
  last_commit_msg: Merge pull request #1002 from siege-analytics/feat/816-notebook-coverage
  repo_path: /Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities


## 4. Branch Inventory

See all local and remote branches with their latest activity.
Stale branches with no recent commits are cleanup candidates.

In [4]:
branch_info = get_branch_info(repo_path=repo_root)

print(f"Local branches: {len(branch_info.get('local', []))}")
for b in branch_info.get('local', [])[:10]:
    current = ' <-' if b.get('is_current') else ''
    print(f"  {b.get('name', 'unknown')}{current}")

print(f"\nRemote branches: {len(branch_info.get('remote', []))}")
for b in branch_info.get('remote', [])[:5]:
    print(f"  {b.get('name', 'unknown')}")

Local branches: 0

Remote branches: 0


## Key Patterns

- **get_repository_status** — single-call snapshot of the entire repo state
- **get_commit_history + categorize_commits** — conventional-commit-aware history analysis
- **analyze_branch_status** — ahead/behind tracking against main
- **get_branch_info** — local + remote branch inventory for cleanup

All read-only — none of these functions modify the repository.